The code here is mainly adapted from https://github.com/structuredllm/syncode/blob/main/notebooks/example_date.ipynb

In [ ]:
%pip -q install "syncode==0.4.13" "transformers==4.51.3" "accelerate==1.15.0" "hf_xet"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.7/205.7 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.

In [ ]:
import operator
from syncode import Syncode
from syncode.larkm import Lark

grammar = r'''
start: number OP number ";"
?number: SCIENTIFIC | NUMBER
OP: "+" | "-" | "*" | "/"
NUMBER: /[+-]?([0-9]+(\.[0-9]*)?|\.[0-9]+)/
SCIENTIFIC: NUMBER /[eE][+-]?[0-9]+/
%import common.WS
%ignore WS
'''

model_name = "microsoft/phi-2"
syn_llm = Syncode(model=model_name, grammar=grammar, parse_output_only=True,
                  max_new_tokens=80)
parser = Lark(grammar)

prompt = '''Extract two operands in calculation order and one operator.
Output only a op b, don't include the result.
Copy numbers exactly, preserving signs, decimal points and e/E exponents.
Do not calculate or expand exponents.
For subtraction, b subtracted from a means a - b.
For division, b divided by a means b / a.
For + and *, keep the query order.

Query: What is 2. plus .5?
Extraction: 2. + .5;
Query: Subtract -406.607 from 919.714.
Extraction: 919.714 - -406.607;
Query: Multiply -1.2e3 by 4.5E-2.
Extraction: -1.2e3 * 4.5E-2;
Query: Divide -8.4 by 2.0.
Extraction: -8.4 / 2.0;
Query: {query}
Extraction:'''

operations = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}

def run_syncode(textual_query):
    a, op, b = parser.parse(syn_llm.infer(prompt.format(query=textual_query))[0]).children
    a, b = float(a), float(b)
    return [a, b], str(op), float(operations[op](a, b))

[2026-10-29 03:38:18,451-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

[2026-10-29 03:39:25,785-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_1357703025_50257.pkl exist: False
[2026-10-29 03:39:25,786-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_1357703025_50257.pkl.
[2026-10-29 03:39:26,334-syncode.mask_store.fsm_set] - 5 FSMs with 32 states initialized in 0.01 seconds
[2026-10-29 03:39:26,334-syncode.mask_store.mask_store] - Ignore whitespace is True
[2026-10-29 03:39:26,339-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 25 to 11


100%|██████████| 32/32 [00:07<00:00,  4.55it/s]


[2026-10-29 03:39:33,417-syncode.mask_store.mask_store] - Time taken to create mask store: 7.61 seconds


In [ ]:
run_syncode('What is 327. multiplied by 11.0?')

([327.0, 11.0], '*', 3597.0)

In [ ]:
run_syncode('What is 45.1 plus 23.54?')

([45.1, 23.54], '+', 68.64)

In [ ]:
run_syncode('What is 120.4 divided by 4.0?')

([120.4, 4.0], '/', 30.1)